## Processing RGB-encoded GeoTIFF masks from IRIS

After producing the AI-assisted cloud masks in IRIS, further processing is needed before use in the comparison. The IRIS reference masks have no georeferencing, meaning no spatial information is assigned. They are RGB-encoded, with "no cloud" pixels having a value of 255 and "cloud" pixels set as 0. This is the exact opposite of how the masks are labeled for all three algorithms being compared. Additionally, data type is int64. For these reasons, we need to process these GeoTIFF files prior to analysis.

### Imports

In [ ]:
import getpass
import time
import glob
from pathlib import Path
import logging
import yaml  # For reading configuration files.

import ee
import ee.batch
import rasterio  # For reading and writing geospatial raster data.
import numpy as np  # For numerical operations on arrays.


try:
    from google.colab import drive  # For mounting Google Drive in Colab.
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False


logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('mask_generation')

def initialize_earth_engine():
    """Authenticate and initialize Earth Engine with user-provided project ID."""
    project = getpass.getpass('Enter your EE Project ID: ').strip()
    if not project:
        raise ValueError('Earth Engine project ID is required.')

    try:
        ee.Initialize(project=project)
        logger.info('Earth Engine initialized with existing credentials.')
    except Exception:
        logger.info('Earth Engine authentication required; opening auth flow...')
        ee.Authenticate()
        ee.Initialize(project=project)
        logger.info('Earth Engine authenticated and initialized successfully.')
    return project

project_id = initialize_earth_engine()

if IN_COLAB and drive is not None:
    try:
        drive.mount('/content/gdrive/My Drive')
    except Exception as exc:
        logger.warning('Google Drive mount failed: %s', exc)
else:
    logger.info('Not running in Colab; skipping Google Drive mount.')

In [ ]:
def load_config() -> tuple[str, dict]:
    """Load config.yaml from likely locations and validate required keys.
    Returns:
        A tuple containing the path to the config file and the loaded config dictionary.
    Raises:
        FileNotFoundError: If no config.yaml file is found in expected locations.
        ValueError: If the config file contains invalid YAML.
        KeyError: If required keys are missing from the config.
    """
    candidates = []
    if IN_COLAB:
        candidates.append(Path('/content/gdrive/My Drive/config.yaml'))

    # Local fallbacks for running outside Colab.
    candidates.extend([
        Path.cwd() / 'config.yaml',
        Path.cwd() / 'config' / 'config.yaml',
        Path.cwd().parent / 'config.yaml',
        Path.cwd().parent / 'config' / 'config.yaml'
    ])

    checked = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.add(str(candidate))

        if not candidate.exists():
            continue

        try:
            with open(candidate, 'r') as f:
                cfg = yaml.safe_load(f) or {}
        except yaml.YAMLError as exc:
            raise ValueError(f'Invalid YAML in config file: {candidate}') from exc

        required_keys = ['polygon', 'date_ranges', 'thresholds', 'paths']
        missing = [key for key in required_keys if key not in cfg]
        if missing:
            raise KeyError(
                f'Missing required config keys in {candidate}: {missing}'
            )

        return str(candidate), cfg

    raise FileNotFoundError(
        'Could not find config.yaml in expected locations. '
        'Update the path or place config.yaml in Drive/local config folder.'
    )

config_path, config = load_config()
logger.info('Loaded config from %s', config_path)

### Define region and dates for export

In [ ]:
# Define area of interest: Sandusky Bay, Ohio.
AOI = ee.Geometry.Polygon(config['polygon'])
logger.info('Area of interest defined with %d vertices.', len(config['polygon']))

# Define date ranges for image collection as list of tuples (start, end).
# Start and end dates should be in 'YYYY-MM-DD' format. End date is exclusive.   
date_ranges = [(dr['start'], dr['end']) for dr in config['date_ranges']]
logger.info('Defined %d date ranges for image collection.', len(date_ranges))

### Export EE image to Google Drive

In [ ]:
# Use with export function to visualize the cloud mask in black and white.

def cloud_mask_vis(image):
    # Define the range of pixel values that will be mapped to 0-255.
    vis = image.visualize(**{
        'palette': ['black', 'white'],
        "min": 0,     # Pixel values at or below this will be displayed as black.
        "max": 0.4    # Pixel values at or above this will be displayed as white.
    })
    return vis

In [ ]:
# Export to a folder in Google Drive. If raw mask is used as input for
# the image parameter, the output will be binary (values strictly 0 or 1). Else,
# if the cloud_mask_vis function is used with the mask, the output will contain values from 0 to 255.

def export_image(
    mask,
    description,
    folder='Results',
    region=AOI,
    scale=10,
    crs='EPSG:4326',
    maxPixels=1_000_000_000,
    timeout_minutes=120,
    poll_seconds=60,
    raise_on_error=False
) -> dict:
    """Export an Earth Engine image and monitor task state with timeout handling.
    Args:
        mask: An ee.Image object representing the image to export.
        description: A string description for the export task, used as the filename.
        folder: A string representing the Google Drive folder to export to (default 'Results').
        region: An ee.Geometry object defining the export region (default AOI).
        scale: An integer representing the resolution in meters per pixel (default 10).
        crs: A string representing the coordinate reference system for export (default 'EPSG:4326').
        maxPixels: An integer representing the maximum number of pixels allowed in the export (default 1 billion).
        timeout_minutes: An integer representing the maximum time to wait for export completion in minutes (default 120).
        poll_seconds: An integer representing the interval in seconds to check the export task status (default 60).
        raise_on_error: A boolean indicating whether to raise exceptions on errors (default False). 
                        If False, errors will be logged and returned in the status dictionary.
    Returns:
        A dictionary containing the final status of the export task, including 'state' and any 'error_message' if applicable.
    Raises:
        RuntimeError: If the export task fails and raise_on_error is True.
        TimeoutError: If the export task times out and raise_on_error is True.
    Note:
        - The function starts an export task and continuously polls its status until it reaches 
        a terminal state ('COMPLETED', 'FAILED', 'CANCELLED') or exceeds the specified timeout.
        - If the task fails or is cancelled, the error message from the task status will be 
        logged and included in the returned status dictionary. If raise_on_error is True, 
        a RuntimeError will be raised with the error details.
        - If the task exceeds the timeout, it will attempt to cancel the task and return 
        a status indicating a timeout. If raise_on_error is True, a TimeoutError will be raised.
    """
    try:
        task = ee.batch.Export.image.toDrive(
            image=mask,
            description=description,    # Filename.
            # The Google Drive Folder that the export will reside in. Note:
            # (a) if the folder name exists at any level, the output is written to it,
            # (b) if duplicate folder names exist, output is written to the most recently modified folder,
            # (c) if the folder name does not exist, a new folder will be created at the root, and
            # (d) folder names with separators (e.g. 'path/to/file') are interpreted as literal strings,
            # not system paths. Defaults to Drive root.
            folder=folder,
            region=region,
            scale=scale,    # Resolution in meters per pixel. Defaults to 1000.
            crs=crs,      # Default is global GCS; EPSG:32617 is specific to Ohio.
            maxPixels=maxPixels
        )
        task.start()
        logger.info('Export started: %s (folder=%s)', description, folder)
    except Exception as exc:
        logger.error('Failed to start export for %s: %s', description, exc)
        if raise_on_error:
            raise
        return {'state': 'START_FAILED', 'error_message': str(exc)}

    start_time = time.time()
    last_state = None
    terminal_states = {'COMPLETED', 'FAILED', 'CANCELLED'}

    while True:
        try:
            status = task.status()
        except Exception as exc:
            logger.error('Failed to read task status for %s: %s', description, exc)
            if raise_on_error:
                raise
            return {'state': 'STATUS_ERROR', 'error_message': str(exc)}

        state = status.get('state', 'UNKNOWN')
        if state != last_state:
            logger.info('Task %s state: %s', description, state)
            last_state = state

        if state in terminal_states:
            if state == 'COMPLETED':
                logger.info('Export completed: %s', description)
            else:
                logger.error(
                    'Export ended with state %s for %s. Details: %s',
                    state,
                    description,
                    status.get('error_message', 'No error message provided.')
                )
                if raise_on_error:
                    raise RuntimeError(
                        f"Export failed for {description}: {status.get('error_message', state)}"
                    )
            return status

        elapsed_minutes = (time.time() - start_time) / 60
        if elapsed_minutes > timeout_minutes:
            logger.error('Export timed out after %.1f minutes: %s', elapsed_minutes, description)
            try:
                task.cancel()
                logger.warning('Timed-out task cancelled: %s', description)
            except Exception as exc:
                logger.warning('Could not cancel task %s: %s', description, exc)

            timeout_status = {'state': 'TIMEOUT', 'description': description}
            if raise_on_error:
                raise TimeoutError(f'Export timed out for {description}.')
            return timeout_status

        time.sleep(poll_seconds)

### Write binary mask files to Drive (necessary for comparison)

In [ ]:
# Choose any cloud mask file from any of three algorithms to copy metadata.
# This is crucial to ensure that the reference masks are properly georeferenced.
sample_mask_path = Path(config['paths']['sample_mask_path'])
logger.info('Using sample mask for metadata reference: %s', sample_mask_path)

with rasterio.open(sample_mask_path) as ref:
    ref_meta = ref.meta.copy()

# Output cloud mask files from the IRIS program need to be uploaded to this directory
# in Google Drive before running the code below to convert them to georeferenced binary
# masks that align with the other two algorithms and can be used for evaluation.
iris_files = glob.glob(f'{Path(config["paths"]["reference_masks_root"])}/*.tif')
logger.info('Found %d IRIS mask files to process in: %s', len(iris_files), config["paths"]["reference_masks_root"])

for iris_input_path in iris_files:
    iris_output_path = iris_input_path.replace(".tif", "_processed.tif")

    with rasterio.open(iris_input_path) as mask:
        # Mask information is in 3rd band (layer).
        iris_data = np.array(mask.read(3))
        # Convert 255 to 0 (no cloud), else 1 (cloud) and convert data type
        # to align with masks produced by the three algorithms.
        iris = np.where(iris_data == 255, 0, 1).astype(rasterio.uint8)

    # Open each mask file and write the newly processed
    # and georeferenced version to the output path.
    with rasterio.open(iris_output_path, "w", **ref_meta) as dest:
        # Write the NumPy array to the first band (band index starts at 1)
        dest.write(iris, 1)
        logger.info('Unique values in saved mask: %s', np.unique(iris))
        logger.info('CRS: %s', dest.crs)
        logger.info('Transform: %s', dest.transform)


    logger.info('Saved %s to: %s', iris_input_path, iris_output_path)

### Export RGB mask files to Drive (optional)

In [ ]:
# For this section to work, you must first download the processed IRIS mask files
# locally to your device, then manually upload them in the Assets tab of GEE.

if not project_id:
    raise ValueError("project_id is empty. Set project_id before exporting masks.")

# Specify the path to your folder or image collection.
asset_path = f"projects/{project_id}/assets/"

# List assets in the specified path with defensive error handling.
try:
    asset_list = ee.data.listAssets(asset_path)
except Exception as exc:
    raise RuntimeError(f"Failed to list assets at {asset_path}: {exc}") from exc

assets = asset_list.get("assets", [])
if not assets:
    print(f"No assets found at: {asset_path}")
else:
    # Iterate through the list, print asset details, and export each as a mask file.
    for asset in assets:
        asset_id = asset.get("id")
        if not asset_id:
            logger.info(f"Skipping malformed asset record: {asset}")
            continue

        try:
            iris_mask = ee.Image(asset_id)

            # Rename the files to avoid overwriting when exporting.
            asset_name = asset_id.split("/")[-1]
            export_name = asset_name.replace("processed", "mask")

            export_image(cloud_mask_vis(iris_mask), export_name, "reference")
            logger.info('Export started for: %s -> %s', asset_id, export_name)
        except Exception as exc:
            logger.error('Failed to export asset "%s": %s', asset_id, exc)

### Validation

In [ ]:
def check_mask(binary_mask) -> None:
    """
    Checks the properties of a binary mask.
    Args:
        binary_mask: A numpy array representing the binary mask.
    """
    # Check unique values.
    unique_vals = np.unique(binary_mask)
    logger.info(f"Unique values: {unique_vals}")
    logger.info(f"Number of unique values: {len(unique_vals)}")

    # Confirm binary format (only 0s and 1s).
    is_binary = set(unique_vals).issubset({0, 1})
    logger.info(f"Is binary (0,1): {is_binary}")

    # Check pixel values and percentage cloudy.
    logger.info(f"Total pixels: {binary_mask.size}")
    logger.info(f"Cloud pixels (value=1): {np.sum(binary_mask == 1)}")
    logger.info(f"Clear pixels (value=0): {np.sum(binary_mask == 0)}")
    logger.info(f"Percentage cloudy: {(np.sum(binary_mask == 1) / binary_mask.size) * 100:.2f}%")

    # Check data type.
    logger.info(f"Data type: {binary_mask.dtype}")

In [ ]:
for ref_mask in glob.glob(f'{Path(config["paths"]["reference_masks_root"])}/*.tif'):
    with rasterio.open(ref_mask) as mask:
        data = np.array(mask.read())

    check_mask(data)